# Ciclo 04 - Modelo Final (pipeline modularizado)

Este notebook **não re-implementa** o pipeline: ele **importa as classes** do pacote `turnover` (criadas no Ciclo 04) e demonstra o fluxo ponta a ponta:

1. Carga e validação dos dados (contrato de colunas).
2. Split treino / validação / teste.
3. **Combina treino + validação** para o modelo final (a validação já cumpriu seu papel nos ciclos 1-3).
4. Treina o `TurnoverPipeline` (feature engineering + pré-processamento + Logistic Regression).
5. Aplica no **teste** (lacrado até agora) e avalia **uma única vez**.
6. Serializa o pipeline treinado com `joblib` para produção.

> O código de verdade vive em `src/turnover/`. Aqui é documentação executável do uso do pacote.

## 1.0 Imports das classes do pacote

In [ ]:
import sys
from pathlib import Path

# garante que 'src' esteja no path para importar o pacote turnover
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from sklearn.model_selection import train_test_split

from turnover.config import (
    DATA_PATH, PIPELINE_PATH, TARGET, SEED,
    TEST_SIZE, VAL_SIZE, BEST_THRESHOLD,
)
from turnover.data import load_csv, validate
from turnover.pipeline import TurnoverPipeline
from turnover.evaluate import ml_metrics, best_threshold_by_f1

## 2.0 Carga e validação dos dados

`load_csv` + `validate` vêm do pacote. O `validate` checa o contrato de colunas (`schema.py`) e falha cedo se o CSV não tiver as colunas obrigatórias.

In [ ]:
df = validate(load_csv(DATA_PATH))
print(f'{df.shape[0]} linhas, {df.shape[1]} colunas')
df.head()

## 3.0 Split treino / validação / teste

Mesmos `random_state` e proporções dos ciclos anteriores (vêm do `config.py`), estratificado no alvo para preservar a proporção de turnover.

In [ ]:
df_train_full, df_test = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[TARGET], random_state=SEED
)
df_train, df_val = train_test_split(
    df_train_full, test_size=VAL_SIZE, stratify=df_train_full[TARGET], random_state=SEED
)

print(f'Treino: {len(df_train)} | Validação: {len(df_val)} | Teste: {len(df_test)}')

## 4.0 Modelo final: treino + validação combinados

Como a validação já cumpriu o papel de ajudar nas decisões (features e hiperparâmetros nos ciclos 1-3), agora ela vira dado de treino. O modelo final aprende com **mais dados**. O teste continua intocado.

In [ ]:
df_train_final = pd.concat([df_train, df_val], ignore_index=True)
print(f'Treino final (treino + val): {len(df_train_final)} linhas')

# treina o pipeline inteiro: feature engineering -> pré-processamento -> LR
pipeline = TurnoverPipeline().fit(df_train_final)
print('Pipeline treinado.')
print('Nº de features após o pré-processamento:', len(pipeline.prep.feature_names_))

## 5.0 Avaliação no conjunto de TESTE

Primeira e única vez que o teste é usado. O `predict_proba` do pipeline aplica **exatamente** a mesma sequência de transformações do treino (sem leakage, sem descompasso de colunas).

In [ ]:
# alvo do teste em 0/1 (mesma conversão usada internamente pelo pipeline)
y_test = TurnoverPipeline._as_binary_target(df_test[TARGET])

proba_test = pipeline.predict_proba(df_test)
ml_metrics('Logistic Regression - Teste', y_test, proba_test, BEST_THRESHOLD)

### 5.1 Ajuste de threshold (opcional)

O limiar 0.5 é arbitrário. `best_threshold_by_f1` varre a curva precision-recall e retorna o corte que maximiza o F1. Para priorizar recall (pegar mais quem vai sair), baixe o limiar conscientemente, aceitando perda de precision.

In [ ]:
thr, f1 = best_threshold_by_f1(y_test, proba_test)
print(f'Melhor threshold (F1): {thr:.4f}  |  F1: {f1:.4f}')

# métricas no threshold ótimo
ml_metrics('LR - Teste (threshold ótimo)', y_test, proba_test, thr)

## 6.0 Serialização do pipeline (artefato de produção)

Salva o pipeline **inteiro já treinado** (feature engineering + pré-processamento + modelo) em um arquivo `.joblib`. É esse artefato que o `predict.py` e o app Streamlit vão carregar — treino e produção usam o mesmo código.

> Segurança: carregar `.joblib` executa código na desserialização. Só carregue artefatos gerados pelo próprio projeto, nunca arquivos de terceiros.

In [ ]:
pipeline.save(PIPELINE_PATH)
print(f'Pipeline salvo em: {PIPELINE_PATH}')

# demonstra o carregamento e uso (como será em produção)
pipeline_carregado = TurnoverPipeline.load(PIPELINE_PATH)
proba_check = pipeline_carregado.predict_proba(df_test.head())
print('Previsões do pipeline recarregado (primeiras linhas):', proba_check.round(4))

## Resumo

- O pipeline inteiro virou um pacote em classes (`src/turnover/`), com padrão `fit`/`transform` que elimina leakage e descompasso de colunas por construção.
- O modelo final (Logistic Regression) foi treinado em treino + validação e avaliado no teste uma única vez.
- O artefato `.joblib` fecha o ciclo treino → produção: o mesmo pipeline é reusado pelo `predict.py` e pelo app Streamlit.
- O entrypoint de linha de comando equivalente é o `train.py` na raiz (`uv run train.py`).